# 10 — Hot vs reference day mobility comparison

This notebook compares intra-Sevilla mobility on selected hot and reference weekdays.

The analysis is **descriptive, not causal**. The selected dates are a first-stage comparison set; a final study should attach observed hourly weather/UTCI to each date rather than rely only on calendar classification.

Hot dates:
- 18, 19, 23, 24, 25 July 2024

Reference dates:
- 27, 28 June and 2, 3 July 2024

In [ ]:
from pathlib import Path

# The notebooks_v02 folder is expected to sit directly inside the repository.
# If you open Jupyter from another working directory, set PROJECT_ROOT manually.
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks_v02" else HERE

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
OUTPUTS = PROJECT_ROOT / "outputs"

for folder in [DATA_RAW, DATA_INTERIM, OUTPUTS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MOBILE_INTERIM = PROJECT_ROOT / "data" / "mobile" / "interim"
MOBILE_PROCESSED = PROJECT_ROOT / "data" / "mobile" / "processed"
MOBILE_PROCESSED.mkdir(parents=True, exist_ok=True)

od = pd.read_parquet(
    MOBILE_INTERIM / "sevilla_od_heat_reference_sample.parquet"
)

zones = gpd.read_file(
    MOBILE_INTERIM / "sevilla_districts_v2.gpkg"
)

od["date"] = pd.to_datetime(od["date"])

## 1. Label hot/reference dates and isolate intra-Sevilla trips

In [ ]:
hot_dates = pd.to_datetime([
    "2024-07-18",
    "2024-07-19",
    "2024-07-23",
    "2024-07-24",
    "2024-07-25",
])

reference_dates = pd.to_datetime([
    "2024-06-27",
    "2024-06-28",
    "2024-07-02",
    "2024-07-03",
])

od["thermal_period"] = np.select(
    [
        od["date"].isin(hot_dates),
        od["date"].isin(reference_dates),
    ],
    ["hot", "reference"],
    default="other",
)

sevilla_ids = set(zones["id"].astype(str))

od["id_origin"] = od["id_origin"].astype(str)
od["id_destination"] = od["id_destination"].astype(str)

intra = od[
    od["id_origin"].isin(sevilla_ids)
    & od["id_destination"].isin(sevilla_ids)
].copy()

print("Intra-Sevilla rows:", f"{len(intra):,}")

## 2. Compare daily mobility

In [ ]:
daily_totals = (
    intra.groupby(
        ["date", "thermal_period"],
        as_index=False,
    )
    .agg(n_trips=("n_trips", "sum"))
)

display(
    daily_totals.groupby("thermal_period")["n_trips"]
    .agg(
        days="count",
        mean="mean",
        median="median",
        std="std",
        minimum="min",
        maximum="max",
    )
)

## 3. Matched-day diagnostic

In [ ]:
date_pairs = {
    "2024-07-18": "2024-06-27",
    "2024-07-19": "2024-06-28",
    "2024-07-23": "2024-07-02",
    "2024-07-24": "2024-07-03",
    "2024-07-25": "2024-06-27",  # Reused Thursday for PoC only.
}

daily_lookup = daily_totals.set_index("date")["n_trips"]
rows = []

for hot_date, ref_date in date_pairs.items():
    hot_date = pd.Timestamp(hot_date)
    ref_date = pd.Timestamp(ref_date)

    hot_trips = daily_lookup.loc[hot_date]
    ref_trips = daily_lookup.loc[ref_date]

    rows.append({
        "hot_date": hot_date,
        "reference_date": ref_date,
        "hot_trips": hot_trips,
        "reference_trips": ref_trips,
        "change_pct": (
            (hot_trips - ref_trips)
            / ref_trips
            * 100
        ),
    })

pair_comparison = pd.DataFrame(rows)
display(pair_comparison)

print(
    "Mean paired mobility change:",
    f"{pair_comparison['change_pct'].mean():.1f}%"
)

## 4. Compare hourly profiles correctly at the day-hour level

In [ ]:
hourly_daily = (
    intra.groupby(
        ["date", "thermal_period", "hour"],
        as_index=False,
    )
    .agg(total_trips=("n_trips", "sum"))
)

hourly_compare = (
    hourly_daily.groupby(
        ["thermal_period", "hour"],
        as_index=False,
    )
    .agg(mean_trips=("total_trips", "mean"))
)

hourly_wide = (
    hourly_compare
    .pivot(
        index="hour",
        columns="thermal_period",
        values="mean_trips",
    )
    .reset_index()
)

hourly_wide["change_pct"] = (
    (hourly_wide["hot"] - hourly_wide["reference"])
    / hourly_wide["reference"]
    * 100
)

fig, ax = plt.subplots(figsize=(9, 5))

for period, group in hourly_compare.groupby("thermal_period"):
    ax.plot(
        group["hour"],
        group["mean_trips"],
        marker="o",
        label=period,
    )

ax.set_xlabel("Hour")
ax.set_ylabel("Mean estimated trips")
ax.set_title("Hourly intra-Sevilla mobility: hot vs reference days")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Summarize morning, afternoon, and evening change

In [ ]:
def mean_period_change(df, start_hour, end_hour):
    return df.loc[
        df["hour"].between(start_hour, end_hour),
        "change_pct",
    ].mean()


period_summary = pd.DataFrame({
    "period": [
        "Morning 06–11",
        "Afternoon 12–18",
        "Evening 19–23",
    ],
    "change_pct": [
        mean_period_change(hourly_wide, 6, 11),
        mean_period_change(hourly_wide, 12, 18),
        mean_period_change(hourly_wide, 19, 23),
    ],
})

display(period_summary)

## 6. District-level origin mobility change

In [ ]:
district_daily = (
    intra.groupby(
        ["date", "thermal_period", "id_origin"],
        as_index=False,
    )
    .agg(trips=("n_trips", "sum"))
)

district_period = (
    district_daily.groupby(
        ["id_origin", "thermal_period"],
        as_index=False,
    )
    .agg(mean_daily_trips=("trips", "mean"))
)

district_wide = (
    district_period
    .pivot(
        index="id_origin",
        columns="thermal_period",
        values="mean_daily_trips",
    )
    .reset_index()
)

district_wide["change_pct"] = (
    (district_wide["hot"] - district_wide["reference"])
    / district_wide["reference"]
    * 100
)

zone_lookup = zones[
    ["id", "name", "population"]
].copy()

zone_lookup["id"] = zone_lookup["id"].astype(str)

zone_lookup = zone_lookup.rename(columns={
    "id": "id_origin",
    "name": "district_name",
    "population": "district_population",
})

district_results = (
    district_wide[
        ["id_origin", "reference", "hot", "change_pct"]
    ]
    .merge(
        zone_lookup,
        on="id_origin",
        how="left",
    )
    [
        [
            "id_origin",
            "district_name",
            "district_population",
            "reference",
            "hot",
            "change_pct",
        ]
    ]
    .sort_values("change_pct")
)

display(district_results)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.barh(
    district_results["district_name"],
    district_results["change_pct"],
)

ax.axvline(0, linewidth=1)
ax.set_xlabel("Hot-day mobility change relative to reference days (%)")
ax.set_ylabel("Origin district")
ax.set_title("District-level intra-Sevilla mobility change")

plt.tight_layout()
plt.show()

## 7. Save the district comparison

In [ ]:
district_results.to_parquet(
    MOBILE_PROCESSED / "sevilla_district_hot_reference_mobility.parquet",
    index=False,
)

## Interim interpretation

In the initial comparison, hot days showed lower intra-Sevilla mobility, with larger reductions after midday. These differences are descriptive only: calendar matching does not control for all confounders and does not establish a causal heat effect.